# Nonlinear Conjugate Gradient and L-BFGS for Binary MNIST Classification

Train a binary logistic regression classifier on MNIST digits 0 and 1 using Fletcher–Reeves (NCG-FR), Polak–Ribière+ (NCG-PR+) and limited-memory BFGS. Each optimizer starts from zero parameters and uses Armijo backtracking. The experiment compares initial trial lengths, objective reduction, gradient norm and classification accuracy.

[Technical notes](../docs/logistic_regression_optimization.md) · [Saved figures](../results/logistic_regression_optimization/)

In [1]:
import os
import time
import json
import math
import numpy as np
import matplotlib.pyplot as plt

## Experimental overview

All three optimizers minimize the same unregularized sum of cross-entropy losses. The model has 784 pixel weights and one bias.

## Numerical utilities and classification rule

Pixels are represented by a 784-dimensional vector. The sigmoid maps the score $z=w^Tx+b$ to a probability. The implementation deliberately evaluates labels using the **raw score threshold 0.5**, not a probability threshold of 0.5 (which would correspond to a raw score of zero). Accuracy is one minus the mean absolute binary-label error.

In [2]:
def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def sigmoid(z: np.ndarray) -> np.ndarray:
    # stable sigmoid
    z = np.clip(z, -50.0, 50.0)
    return 1.0 / (1.0 + np.exp(-z))

def pack_theta(w: np.ndarray, b: float) -> np.ndarray:
    return np.concatenate([w.reshape(-1), np.array([b], dtype=w.dtype)])

def unpack_theta(theta: np.ndarray):
    w = theta[:-1]
    b = float(theta[-1])
    return w, b

def predict_proba(theta: np.ndarray, X: np.ndarray) -> np.ndarray:
    w, b = unpack_theta(theta)
    return sigmoid(X @ w + b)

def predict_label(theta: np.ndarray, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
    # y' = 1 if w^T x + b >= 0.5 else 0 (as in prompt)
    w, b = unpack_theta(theta)
    scores = X @ w + b
    return (scores >= threshold).astype(np.int64)

def accuracy_from_labels(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    # prompt shows accuracy = sum |y - y'| / n (that's actually error rate)
    # We'll report both:
    # - error_rate = mean(|y - y_pred|)
    # - acc = 1 - error_rate
    err = float(np.mean(np.abs(y_true - y_pred)))
    return 1.0 - err

## Logistic objective and gradient

$$L(\theta)=-\sum_i[y_i\log p_i+(1-y_i)\log(1-p_i)],\quad p_i=\sigma(w^Tx_i+b).$$

The gradient is $[X^T(p-y),\mathbf1^T(p-y)]$. The routine supports optional weight-only L2 regularization, but the experiment uses zero regularization. Probability clipping and a small logarithm offset are used for numerical stability.

In [3]:
def loss_and_grad(theta: np.ndarray, X: np.ndarray, y: np.ndarray, reg: float = 0.0):
    # y in {0,1}
    w, b = unpack_theta(theta)
    n = X.shape[0]
    p = sigmoid(X @ w + b)
    eps = 1e-12
    L = -np.sum(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))

    # gradient
    r = (p - y)  # shape (n,)
    gw = X.T @ r
    gb = float(np.sum(r))

    # optional L2 regularization on w only
    if reg > 0:
        L = L + 0.5 * reg * np.dot(w, w)
        gw = gw + reg * w

    g = pack_theta(gw, gb)
    return float(L), g


## Armijo backtracking

Accept $\alpha=\alpha_0\rho^j$ when $L(\theta+\alpha d)\le L(\theta)+c_1\alpha g^Td$. Non-descent directions fall back to $-g$. The search is capped at 50 trials and returns an unsuccessful zero step if no trial is accepted.

In [4]:
def backtracking_armijo(fg, theta, f0, g0, d, alpha0=1.0, rho=0.5, c1=1e-4, max_ls=50):
    gtd = float(np.dot(g0, d))
    if gtd >= 0:
        # not a descent direction; fallback to steepest descent
        d = -g0
        gtd = float(np.dot(g0, d))
    alpha = float(alpha0)
    for _ in range(max_ls):
        theta_new = theta + alpha * d
        f_new, g_new = fg(theta_new)
        if f_new <= f0 + c1 * alpha * gtd:
            return alpha, theta_new, f_new, g_new, True
        alpha *= rho
    # failed
    theta_new = theta
    return 0.0, theta_new, f0, g0, False


## Nonlinear conjugate gradient

$$d_{k+1}=-g_{k+1}+\beta_kd_k,$$

with Fletcher–Reeves $\beta_k=\|g_{k+1}\|^2/\|g_k\|^2$ and PR+ $\beta_k=\max(0,g_{k+1}^T(g_{k+1}-g_k)/\|g_k\|^2)$. Restart the direction when it is not descending.

In [5]:
def nlcg_optimize(method: str, X, y, alpha0_list, out_dir,
                  max_iter=200, tol=1e-6, rho=0.5, c1=1e-4, reg=0.0):
    assert method in ("FR", "PR+")
    ensure_dir(out_dir)

    results = []
    for alpha0 in alpha0_list:
        fg = lambda th: loss_and_grad(th, X, y, reg=reg)

        # init theta = 0
        theta = np.zeros(X.shape[1] + 1, dtype=np.float64)
        f, g = fg(theta)
        d = -g

        hist = {
            "iter": [],
            "loss": [],
            "grad_norm": [],
            "alpha": [],
            "ls_success": []
        }

        t0 = time.time()
        for k in range(max_iter):
            gn = float(np.linalg.norm(g))
            hist["iter"].append(k)
            hist["loss"].append(f)
            hist["grad_norm"].append(gn)

            if gn <= tol:
                hist["alpha"].append(0.0)
                hist["ls_success"].append(True)
                break

            alpha, theta_new, f_new, g_new, ok = backtracking_armijo(
                fg, theta, f, g, d, alpha0=alpha0, rho=rho, c1=c1
            )
            hist["alpha"].append(alpha)
            hist["ls_success"].append(ok)

            # update beta
            denom = float(np.dot(g, g)) + 1e-30
            if method == "FR":
                beta = float(np.dot(g_new, g_new)) / denom
            else:  # PR+
                beta_raw = float(np.dot(g_new, (g_new - g))) / denom
                beta = max(0.0, beta_raw)

            d = -g_new + beta * d

            # ensure descent
            if float(np.dot(g_new, d)) >= 0:
                d = -g_new

            theta, f, g = theta_new, f_new, g_new

        elapsed = time.time() - t0
        results.append((alpha0, theta, hist, elapsed))

        # save per-run plots
        tag = f"{method}_alpha0_{alpha0:g}".replace(".", "p")
        save_hist_plots(hist, os.path.join(out_dir, f"{tag}_loss.png"),
                        os.path.join(out_dir, f"{tag}_gradnorm.png"),
                        title_prefix=f"NCG-{method}, alpha0={alpha0:g}")

        # save hist csv
        save_hist_csv(hist, os.path.join(out_dir, f"{tag}_hist.csv"))

        # save final theta
        np.save(os.path.join(out_dir, f"{tag}_theta.npy"), theta)

        # save meta
        with open(os.path.join(out_dir, f"{tag}_meta.json"), "w", encoding="utf-8") as fmeta:
            json.dump({"method": method, "alpha0": alpha0, "elapsed_sec": elapsed}, fmeta, indent=2)

    return results

## Limited-memory BFGS

Use the two-loop recursion to apply an implicit inverse-Hessian approximation. Retain the latest ten displacement/gradient-difference pairs in the experiment, accepting a pair only when $y_k^Ts_k>10^{-12}$.

In [7]:
def lbfgs_optimize(X, y, alpha0_list, out_dir, m=10,
                   max_iter=200, tol=1e-6, rho=0.5, c1=1e-4, reg=0.0):
    ensure_dir(out_dir)
    results = []

    for alpha0 in alpha0_list:
        fg = lambda th: loss_and_grad(th, X, y, reg=reg)
        theta = np.zeros(X.shape[1] + 1, dtype=np.float64)
        f, g = fg(theta)

        S = []  # s_k
        Y = []  # y_k
        RHO = []  # 1/(y^T s)

        hist = {"iter": [], "loss": [], "grad_norm": [], "alpha": [], "ls_success": []}
        t0 = time.time()

        for k in range(max_iter):
            gn = float(np.linalg.norm(g))
            hist["iter"].append(k)
            hist["loss"].append(f)
            hist["grad_norm"].append(gn)

            if gn <= tol:
                hist["alpha"].append(0.0)
                hist["ls_success"].append(True)
                break

            # direction d = -H_k g via two-loop recursion
            d = -two_loop_recursion(g, S, Y, RHO)

            if float(np.dot(g, d)) >= 0:
                d = -g

            alpha, theta_new, f_new, g_new, ok = backtracking_armijo(
                fg, theta, f, g, d, alpha0=alpha0, rho=rho, c1=c1
            )
            hist["alpha"].append(alpha)
            hist["ls_success"].append(ok)

            s = theta_new - theta
            yy = g_new - g
            ys = float(np.dot(yy, s))

            if ys > 1e-12:
                rho_k = 1.0 / ys
                S.append(s)
                Y.append(yy)
                RHO.append(rho_k)
                if len(S) > m:
                    S.pop(0); Y.pop(0); RHO.pop(0)

            theta, f, g = theta_new, f_new, g_new

        elapsed = time.time() - t0
        results.append((alpha0, theta, hist, elapsed))

        tag = f"LBFGS_m{m}_alpha0_{alpha0:g}".replace(".", "p")
        save_hist_plots(hist, os.path.join(out_dir, f"{tag}_loss.png"),
                        os.path.join(out_dir, f"{tag}_gradnorm.png"),
                        title_prefix=f"L-BFGS (m={m}), alpha0={alpha0:g}")
        save_hist_csv(hist, os.path.join(out_dir, f"{tag}_hist.csv"))
        np.save(os.path.join(out_dir, f"{tag}_theta.npy"), theta)
        with open(os.path.join(out_dir, f"{tag}_meta.json"), "w", encoding="utf-8") as fmeta:
            json.dump({"method": "LBFGS", "m": m, "alpha0": alpha0, "elapsed_sec": elapsed}, fmeta, indent=2)

    return results

def two_loop_recursion(g, S, Y, RHO):
    q = g.copy()
    alpha = []
    for s, y, rho in zip(reversed(S), reversed(Y), reversed(RHO)):
        a = rho * float(np.dot(s, q))
        alpha.append(a)
        q = q - a * y

    if len(S) > 0:
        # scaling of initial Hessian approximation
        y_last = Y[-1]
        s_last = S[-1]
        ys = float(np.dot(y_last, s_last))
        yy = float(np.dot(y_last, y_last)) + 1e-30
        gamma = ys / yy
    else:
        gamma = 1.0

    r = gamma * q

    for (s, y, rho), a in zip(zip(S, Y, RHO), reversed(alpha)):
        b = rho * float(np.dot(y, r))
        r = r + s * (a - b)

    return r


## Experiment artifacts

Save loss and gradient-norm plots, history CSV files, final parameter arrays and timing metadata for each configuration.

In [8]:
def save_hist_plots(hist, loss_path, grad_path, title_prefix=""):
    it = np.array(hist["iter"])
    loss = np.array(hist["loss"])
    gn = np.array(hist["grad_norm"])

    plt.figure()
    plt.plot(it, loss)
    plt.xlabel("iteration")
    plt.ylabel("loss")
    if title_prefix:
        plt.title(f"{title_prefix} - Loss")
    plt.tight_layout()
    plt.savefig(loss_path, dpi=200)
    plt.close()

    plt.figure()
    plt.semilogy(it, np.maximum(gn, 1e-30))
    plt.xlabel("iteration")
    plt.ylabel("||grad||_2")
    if title_prefix:
        plt.title(f"{title_prefix} - Grad norm")
    plt.tight_layout()
    plt.savefig(grad_path, dpi=200)
    plt.close()

def save_hist_csv(hist, path):
    import csv
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["iter", "loss", "grad_norm", "alpha", "ls_success"])
        for i, L, gn, a, ok in zip(hist["iter"], hist["loss"], hist["grad_norm"], hist["alpha"], hist["ls_success"]):
            w.writerow([i, L, gn, a, int(ok)])

## MNIST acquisition and preprocessing

Try Keras MNIST, then OpenML, then a local `mnist.npz`. Retain digits 0 and 1, normalize pixels by 255 and flatten each image. Training and test data retain their separate roles.

In [9]:
def load_mnist():
    # Try Keras
    try:
        from tensorflow.keras.datasets import mnist
        (Xtr, ytr), (Xte, yte) = mnist.load_data()
        return (Xtr, ytr), (Xte, yte)
    except Exception:
        pass

    # Try sklearn OpenML
    try:
        from sklearn.datasets import fetch_openml
        mn = fetch_openml("mnist_784", version=1, as_frame=False)
        X = mn.data.reshape(-1, 28, 28).astype(np.uint8)
        y = mn.target.astype(np.int64)
        # standard split: first 60000 train, rest test
        Xtr, ytr = X[:60000], y[:60000]
        Xte, yte = X[60000:], y[60000:]
        return (Xtr, ytr), (Xte, yte)
    except Exception:
        pass

    # Try local npz
    local = "mnist.npz"
    if os.path.exists(local):
        data = np.load(local)
        (Xtr, ytr) = (data["x_train"], data["y_train"])
        (Xte, yte) = (data["x_test"], data["y_test"])
        return (Xtr, ytr), (Xte, yte)

    raise RuntimeError(
        "Could not load MNIST. Options:\n"
        "1) Install tensorflow and retry (uses keras.datasets.mnist).\n"
        "2) Install scikit-learn and retry (uses fetch_openml).\n"
        "3) Place a local file mnist.npz with keys x_train,y_train,x_test,y_test in this folder."
    )

def filter_digits_0_1(X, y):
    mask = (y == 0) | (y == 1)
    X2 = X[mask]
    y2 = y[mask].astype(np.float64)
    return X2, y2

def preprocess(X):
    # normalize and flatten
    X = X.astype(np.float64) / 255.0
    X = X.reshape(X.shape[0], -1)  # (n,784)
    return X

## Experimental protocol

Use initial trial lengths 1, 0.5, 0.1 and 0.05; maximum 200 iterations, gradient tolerance $10^{-6}$, Armijo reduction 0.5 and sufficient-decrease constant $10^{-4}$. Record both training and test accuracy. Summary loss and gradient are the last logged pre-update values, while accuracy uses the returned parameter vector.

In [10]:
def evaluate(theta, Xtr, ytr, Xte, yte, threshold=0.5):
    yhat_tr = predict_label(theta, Xtr, threshold=threshold)
    yhat_te = predict_label(theta, Xte, threshold=threshold)
    acc_tr = accuracy_from_labels(ytr.astype(np.int64), yhat_tr)
    acc_te = accuracy_from_labels(yte.astype(np.int64), yhat_te)
    return acc_tr, acc_te

def main():
    out_root = "outputs"
    ensure_dir(out_root)

    (Xtr0, ytr0), (Xte0, yte0) = load_mnist()
    Xtr0, ytr0 = filter_digits_0_1(Xtr0, ytr0)
    Xte0, yte0 = filter_digits_0_1(Xte0, yte0)

    Xtr = preprocess(Xtr0)
    Xte = preprocess(Xte0)
    ytr = ytr0
    yte = yte0

    # initial theta = 0 required
    # Explore alpha0 values
    alpha0_list = [1.0, 0.5, 0.1, 0.05]

    # hyperparameters (can edit)
    max_iter = 200
    tol = 1e-6
    rho = 0.5
    c1 = 1e-4
    reg = 0.0
    lbfgs_m = 10

    summary_rows = []

    # NCG-FR
    fr_dir = os.path.join(out_root, "NCG_FR")
    fr_runs = nlcg_optimize("FR", Xtr, ytr, alpha0_list, fr_dir,
                            max_iter=max_iter, tol=tol, rho=rho, c1=c1, reg=reg)
    for alpha0, theta, hist, elapsed in fr_runs:
        acc_tr, acc_te = evaluate(theta, Xtr, ytr, Xte, yte, threshold=0.5)
        summary_rows.append(["NCG-FR", alpha0, len(hist["iter"]), hist["loss"][-1], hist["grad_norm"][-1], acc_tr, acc_te, elapsed])

    # NCG-PR+
    pr_dir = os.path.join(out_root, "NCG_PRplus")
    pr_runs = nlcg_optimize("PR+", Xtr, ytr, alpha0_list, pr_dir,
                            max_iter=max_iter, tol=tol, rho=rho, c1=c1, reg=reg)
    for alpha0, theta, hist, elapsed in pr_runs:
        acc_tr, acc_te = evaluate(theta, Xtr, ytr, Xte, yte, threshold=0.5)
        summary_rows.append(["NCG-PR+", alpha0, len(hist["iter"]), hist["loss"][-1], hist["grad_norm"][-1], acc_tr, acc_te, elapsed])

    # L-BFGS
    lb_dir = os.path.join(out_root, "LBFGS")
    lb_runs = lbfgs_optimize(Xtr, ytr, alpha0_list, lb_dir, m=lbfgs_m,
                             max_iter=max_iter, tol=tol, rho=rho, c1=c1, reg=reg)
    for alpha0, theta, hist, elapsed in lb_runs:
        acc_tr, acc_te = evaluate(theta, Xtr, ytr, Xte, yte, threshold=0.5)
        summary_rows.append([f"L-BFGS(m={lbfgs_m})", alpha0, len(hist["iter"]), hist["loss"][-1], hist["grad_norm"][-1], acc_tr, acc_te, elapsed])

    # Save summary CSV
    import csv
    summary_path = os.path.join(out_root, "summary.csv")
    with open(summary_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["method", "alpha0", "iters", "final_loss", "final_grad_norm", "train_acc", "test_acc", "elapsed_sec"])
        for row in summary_rows:
            w.writerow(row)

    # Save a combined bar-like plot for test accuracy vs method/alpha0 (simple scatter)
    plt.figure()
    xs = np.arange(len(summary_rows))
    test_accs = [r[6] for r in summary_rows]
    plt.plot(xs, test_accs, marker="o", linestyle="none")
    plt.xticks(xs, [f"{r[0]}\nα0={r[1]:g}" for r in summary_rows], rotation=90)
    plt.ylabel("test accuracy")
    plt.tight_layout()
    plt.savefig(os.path.join(out_root, "test_accuracy_all.png"), dpi=200)
    plt.close()

    print("Done. Outputs saved in:", out_root)
    print("Summary:", summary_path)

## Run the optimizer comparison

The code writes its generated artifacts to `outputs/` relative to the working directory.

In [11]:
main()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Done. Outputs saved in: outputs
Summary: outputs/summary.csv


## Fletcher–Reeves convergence curves

Loss and gradient norm for the four initial trial lengths.

![FR_alpha0_1_loss.png](../results/logistic_regression_optimization/FR_alpha0_1_loss.png)![FR_alpha0_1_gradnorm.png](../results/logistic_regression_optimization/FR_alpha0_1_gradnorm.png)![FR_alpha0_0p5_loss.png](../results/logistic_regression_optimization/FR_alpha0_0p5_loss.png)![FR_alpha0_0p5_gradnorm.png](../results/logistic_regression_optimization/FR_alpha0_0p5_gradnorm.png)![FR_alpha0_0p05_loss.png](../results/logistic_regression_optimization/FR_alpha0_0p05_loss.png)![FR_alpha0_0p05_gradnorm.png](../results/logistic_regression_optimization/FR_alpha0_0p05_gradnorm.png)![FR_alpha0_0p1_loss.png](../results/logistic_regression_optimization/FR_alpha0_0p1_loss.png)![FR_alpha0_0p1_gradnorm.png](../results/logistic_regression_optimization/FR_alpha0_0p1_gradnorm.png)

## Polak–Ribière+ convergence curves

Loss and gradient norm for the four initial trial lengths.

![PRplus_alpha0_1_loss.png](../results/logistic_regression_optimization/PRplus_alpha0_1_loss.png)![PRplus_alpha0_0p1_loss.png](../results/logistic_regression_optimization/PRplus_alpha0_0p1_loss.png)![PRplus_alpha0_1_gradnorm.png](../results/logistic_regression_optimization/PRplus_alpha0_1_gradnorm.png)![PRplus_alpha0_0p5_loss.png](../results/logistic_regression_optimization/PRplus_alpha0_0p5_loss.png)![PRplus_alpha0_0p05_loss.png](../results/logistic_regression_optimization/PRplus_alpha0_0p05_loss.png)![PRplus_alpha0_0p5_gradnorm.png](../results/logistic_regression_optimization/PRplus_alpha0_0p5_gradnorm.png)![PRplus_alpha0_0p05_gradnorm.png](../results/logistic_regression_optimization/PRplus_alpha0_0p05_gradnorm.png)![PRplus_alpha0_0p1_gradnorm.png](../results/logistic_regression_optimization/PRplus_alpha0_0p1_gradnorm.png)

## Limited-memory BFGS convergence curves

Loss and gradient norm for the four initial trial lengths.

![LBFGS_m10_alpha0_1_loss.png](../results/logistic_regression_optimization/LBFGS_m10_alpha0_1_loss.png)![LBFGS_m10_alpha0_1_gradnorm.png](../results/logistic_regression_optimization/LBFGS_m10_alpha0_1_gradnorm.png)![LBFGS_m10_alpha0_0p5_loss.png](../results/logistic_regression_optimization/LBFGS_m10_alpha0_0p5_loss.png)![LBFGS_m10_alpha0_0p1_gradnorm.png](../results/logistic_regression_optimization/LBFGS_m10_alpha0_0p1_gradnorm.png)![LBFGS_m10_alpha0_0p5_gradnorm.png](../results/logistic_regression_optimization/LBFGS_m10_alpha0_0p5_gradnorm.png)![LBFGS_m10_alpha0_0p05_loss.png](../results/logistic_regression_optimization/LBFGS_m10_alpha0_0p05_loss.png)![LBFGS_m10_alpha0_0p05_gradnorm.png](../results/logistic_regression_optimization/LBFGS_m10_alpha0_0p05_gradnorm.png)![LBFGS_m10_alpha0_0p1_loss.png](../results/logistic_regression_optimization/LBFGS_m10_alpha0_0p1_loss.png)

## Test accuracy across configurations

![test_accuracy_all.png](../results/logistic_regression_optimization/test_accuracy_all.png)

## Results and interpretation

The numerical table below is transcribed from the accompanying experimental analysis (rounded as reported); the saved notebook plots show the same test-accuracy pattern. The original generated summary CSV was not supplied.

| Method | Initial step | Logged loss | Gradient norm | Train acc. | Test acc. |
|---|---:|---:|---:|---:|---:|
| NCG-FR | 1 | 110.5244 | 15.3659 | 0.9997 | 0.9991 |
| NCG-FR | 0.5 | 110.5241 | 20.7197 | 0.9997 | 0.9995 |
| NCG-FR | 0.1 | 276.3102 | 27.8553 | 0.9992 | 0.9995 |
| NCG-FR | 0.05 | 255.6187 | 22.2671 | 0.9992 | 0.9995 |
| NCG-PR+ | 1 | 442.0963 | 34.6280 | 0.9987 | 1.0000 |
| NCG-PR+ | 0.5 | 442.1825 | 34.5873 | 0.9987 | 1.0000 |
| NCG-PR+ | 0.1 | 1243.3832 | 112.2495 | 0.9964 | 0.9991 |
| NCG-PR+ | 0.05 | 1125.9571 | 91.7196 | 0.9967 | 0.9995 |
| L-BFGS | 1 | 233.0927 | 31.0410 | 0.9993 | 0.9991 |
| L-BFGS | 0.5 | 221.0483 | 23.0236 | 0.9994 | 0.9995 |
| L-BFGS | 0.1 | 248.6792 | 38.1942 | 0.9993 | 0.9986 |
| L-BFGS | 0.05 | 295.7253 | 31.2668 | 0.9991 | 0.9986 |

All configurations record 200 iterations, and none has a final logged gradient norm near the stopping tolerance. Thus the near-perfect binary classification accuracy does not establish numerical convergence. FR with trial step 0.5 yields the lowest reported loss; PR+ with 1 or 0.5 yields the highest reported test accuracy (100%). L-BFGS with 0.5 is the best of its four settings by reported loss and test accuracy. These findings do not establish universal optimizer superiority.

The plateaus in the curves indicate stalled or very small progress; the saved figures alone cannot determine the cause. Clipping/offsets and the implemented gradient can differ from the exact derivative of the stabilized loss at extreme scores; line-search failure also permits zero updates. No validation split or repeated independent runs are reported. Comparing test accuracy across trial steps is descriptive rather than an independent hyperparameter-selection protocol.